# CS2 EXP-3 — CodeBERTa-small-v1 Frozen-Encoder Linear Probe

Rewritten to use **CodeBERTa-small-v1** (84M params, RoBERTa architecture, pretrained on CodeSearchNet source code) instead of NeoBERT-250M. No `trust_remote_code`, no xformers/SwiGLU patches -- standard `transformers` loading.

Default pooling is **mean pooling** over tokens (not CLS): raw CLS-token embeddings from a frozen encoder without a pooling-specific pretraining objective are a known weak sentence representation. Set `base_config.pooling = "cls"` to compare.

## 1. Runtime settings

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "prashant"
REPO_ROOT = Path("/content/DiverseVul--IS-Project")
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DRIVE_ROOT = Path("/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData")
PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

SPLIT_ID = "cs1_project_holdout20_innercv_v1"
NORMALIZED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
OUTER_MANIFEST_PATH = MANIFEST_ROOT / SPLIT_ID / "outer_holdout" / "cs1_outer_project_holdout_manifest.parquet"
INNER_MANIFEST_PATH = MANIFEST_ROOT / SPLIT_ID / "inner_cv" / "cs1_project_grouped_5fold_manifest.parquet"

EXP3_OUTPUT_DIR = OUTPUT_ROOT / "case_study_2" / "exp3_codeberta_linear_probe_v1"
EXP3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_CACHE_DIR = DRIVE_ROOT.parent / "embedding_cache_codeberta"
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = "/content/drive/MyDrive/IntelligentSystemProject/hf_cache"

C_GRID = (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0)

RUN_PROFILE = False
RUN_NESTED_OFFICIAL = False
RUN_CANONICAL_RETRAIN = False
RUN_HOLDOUT_EVAL = False

print("Settings loaded. Output dir:", EXP3_OUTPUT_DIR)

## 2. Mount Google Drive and clone/refresh repository

In [ ]:
from google.colab import drive
import subprocess
import sys


def run_command(command, cwd=None):
    print("$", " ".join(str(x) for x in command))
    subprocess.run(command, check=True, cwd=cwd)


drive.mount("/content/drive")

if not REPO_ROOT.exists():
    run_command(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)])
else:
    run_command(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("\nRepository ready.")
run_command(["git", "-C", str(REPO_ROOT), "log", "-1", "--oneline"])

## 3. Verify GPU runtime

In [ ]:
import torch
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not torch.cuda.is_available():
    raise RuntimeError("EXP-3 requires a GPU runtime (Runtime > Change runtime type > GPU).")

DEVICE = "cuda"
print("CUDA device:", torch.cuda.get_device_name(0))
print("bfloat16 supported:", torch.cuda.is_bf16_supported())
print("Total VRAM: %.2f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 4. Install dependencies

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "torch", "numpy", "pandas", "scikit-learn", "matplotlib",
    "pyarrow", "joblib", "tqdm",
], check=True)
print("Dependencies installed/verified.")

## 5. Patch `models.py` and `exp3/exp3_linear_probe.py` with the CodeBERTa-small-v1 rewrite

Overwrites the two files locally in the cloned checkout with the CodeBERTa versions (no NeoBERT/xformers dependencies, mean pooling, length-bucketed + checkpointed embedding extraction). Commit these back to the repo once validated.

In [ ]:
models_path = SRC_DIR / "case_study_2" / "models.py"
exp3_path = SRC_DIR / "case_study_2" / "exp3" / "exp3_linear_probe.py"

models_path.write_text("\"\"\"\nShared CodeBERTa-small-v1 model utilities for Case Study 2.\n\nShared by:\n  - EXP-3 Frozen Linear Probe\n  - EXP-4 LoRA\n\nCodeBERTa-small-v1 is a standard RoBERTa-architecture encoder pretrained on\nCodeSearchNet source code. Unlike NeoBERT, it needs no trust_remote_code,\nno xformers/SwiGLU compatibility shims, and no custom runtime patches --\nit loads via plain transformers.AutoModel / AutoTokenizer.\n\nThis file must not contain experiment-specific training loops.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom typing import Optional, Dict, Any, List\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom transformers import AutoModel, AutoTokenizer\n\n\nDEFAULT_CODE_MODEL = \"huggingface/CodeBERTa-small-v1\"\nDEFAULT_CODE_TOKENIZER = \"huggingface/CodeBERTa-small-v1\"  # ships its own code-trained BPE tokenizer\n\n\ndef configure_huggingface_cache(hf_cache_dir: Optional[str] = None) -> None:\n    \"\"\"\n    Configure Hugging Face cache and safer transfer behavior for Colab.\n    Should be called before model/tokenizer loading.\n    \"\"\"\n    if hf_cache_dir:\n        hf_cache_dir = str(hf_cache_dir)\n        os.environ.setdefault(\"HF_HOME\", hf_cache_dir)\n        os.environ.setdefault(\"HUGGINGFACE_HUB_CACHE\", str(Path(hf_cache_dir) / \"hub\"))\n\n    os.environ.setdefault(\"HF_HUB_DISABLE_XET\", \"1\")\n    os.environ.setdefault(\"HF_HUB_ENABLE_HF_TRANSFER\", \"0\")\n    os.environ.setdefault(\"HF_HUB_DOWNLOAD_TIMEOUT\", \"120\")\n    os.environ.setdefault(\"HF_HUB_ETAG_TIMEOUT\", \"120\")\n\n\ndef _dtype_from_policy(dtype_policy: str, device: str) -> Optional[torch.dtype]:\n    dtype_policy = (dtype_policy or \"auto\").lower()\n    device = str(device)\n\n    if dtype_policy == \"float16\":\n        return torch.float16 if device == \"cuda\" else torch.float32\n    if dtype_policy == \"bfloat16\":\n        return torch.bfloat16 if device == \"cuda\" and torch.cuda.is_bf16_supported() else torch.float32\n    if dtype_policy == \"float32\":\n        return torch.float32\n    if dtype_policy == \"auto\":\n        if device == \"cuda\" and torch.cuda.is_bf16_supported():\n            return torch.bfloat16\n        if device == \"cuda\":\n            return torch.float32\n        return torch.float32\n\n    raise ValueError(f\"Unknown dtype_policy: {dtype_policy}\")\n\n\ndef load_code_tokenizer(\n    tokenizer_name: str = DEFAULT_CODE_TOKENIZER,\n    hf_cache_dir: Optional[str] = None,\n):\n    configure_huggingface_cache(hf_cache_dir)\n    return AutoTokenizer.from_pretrained(\n        tokenizer_name,\n        use_fast=True,\n        cache_dir=hf_cache_dir,\n    )\n\n\ndef load_code_encoder(\n    model_name: str = DEFAULT_CODE_MODEL,\n    dtype_policy: str = \"auto\",\n    device: Optional[str] = None,\n    freeze: bool = True,\n    hf_cache_dir: Optional[str] = None,\n) -> nn.Module:\n    \"\"\"\n    Load the CodeBERTa encoder. Standard AutoModel loading -- no runtime\n    patches required (unlike NeoBERT's custom architecture).\n    \"\"\"\n    device = device or (\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    configure_huggingface_cache(hf_cache_dir)\n\n    dtype = _dtype_from_policy(dtype_policy, device)\n    is_local_path = Path(str(model_name)).exists()\n\n    kwargs: Dict[str, Any] = {\n        \"cache_dir\": hf_cache_dir,\n        \"local_files_only\": bool(is_local_path),\n    }\n    if dtype is not None:\n        kwargs[\"torch_dtype\"] = dtype\n\n    model = AutoModel.from_pretrained(model_name, **kwargs)\n    model.to(device)\n\n    if freeze:\n        for param in model.parameters():\n            param.requires_grad = False\n        model.eval()\n\n    return model\n\n\ndef mean_pool_last_hidden(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n    \"\"\"\n    Mean-pool token embeddings using the attention mask.\n    Preferred default for frozen probes: raw CLS-token embeddings from a\n    frozen encoder without a pooling-specific pretraining objective are a\n    known weak sentence representation (Reimers & Gurevych, 2019).\n    \"\"\"\n    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)\n    summed = (last_hidden_state * mask).sum(dim=1)\n    denom = mask.sum(dim=1).clamp(min=1.0)\n    return summed / denom\n\n\ndef cls_pool_last_hidden(last_hidden_state: torch.Tensor) -> torch.Tensor:\n    \"\"\"Use first token (<s> / CLS-equivalent) embedding.\"\"\"\n    return last_hidden_state[:, 0, :]\n\n\nclass CodeSequenceClassifier(nn.Module):\n    \"\"\"\n    Shared sequence classifier wrapper for LoRA-style fine-tuning (EXP-4).\n    EXP-3's linear probe normally only uses frozen encoder embeddings\n    extracted separately, but this class is reused wherever an end-to-end\n    trainable classification head is needed.\n    \"\"\"\n\n    def __init__(\n        self,\n        model_name: str = DEFAULT_CODE_MODEL,\n        num_labels: int = 1,\n        freeze_backbone: bool = False,\n        pooling: str = \"mean\",\n        dtype_policy: str = \"auto\",\n        hf_cache_dir: Optional[str] = None,\n    ) -> None:\n        super().__init__()\n        device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n        self.backbone = load_code_encoder(\n            model_name=model_name,\n            dtype_policy=dtype_policy,\n            device=device,\n            freeze=freeze_backbone,\n            hf_cache_dir=hf_cache_dir,\n        )\n        hidden_size = int(self.backbone.config.hidden_size)\n        self.classification_head = nn.Linear(hidden_size, num_labels)\n        self.pooling = pooling\n\n    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, **kwargs: Any) -> torch.Tensor:\n        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kwargs)\n        hidden = outputs.last_hidden_state\n        if self.pooling == \"cls\":\n            pooled = cls_pool_last_hidden(hidden)\n        else:\n            pooled = mean_pool_last_hidden(hidden, attention_mask)\n        logits = self.classification_head(pooled)\n        return logits.squeeze(-1)\n\n\ndef count_trainable_parameters(model: nn.Module) -> Dict[str, int]:\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    total = sum(p.numel() for p in model.parameters())\n    return {\n        \"trainable_parameters\": int(trainable),\n        \"total_parameters\": int(total),\n        \"trainable_percent\": float(100.0 * trainable / max(total, 1)),\n    }\n\n\ndef infer_lora_target_modules(model: nn.Module) -> List[str]:\n    \"\"\"\n    Infer LoRA target module names for a RoBERTa-family encoder (CodeBERTa).\n    RoBERTa self-attention uses separate `query` / `value` Linear layers\n    (not a fused qkv projection like NeoBERT), so this is the expected match.\n    \"\"\"\n    module_names = [name for name, _ in model.named_modules()]\n\n    candidate_sets = [\n        [\"query\", \"value\"],\n        [\"q_proj\", \"v_proj\"],\n        [\"qkv\"],\n        [\"in_proj\"],\n    ]\n\n    for candidates in candidate_sets:\n        if all(any(name.endswith(candidate) or f\".{candidate}\" in name for name in module_names) for candidate in candidates):\n            return candidates\n\n    return [\"query\", \"value\"]\n\n\ndef create_lora_sequence_classifier(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    lora_dropout: float = 0.05,\n    dtype_policy: str = \"auto\",\n    hf_cache_dir: Optional[str] = None,\n):\n    \"\"\"\n    Shared LoRA model creation helper for EXP-4. PEFT is imported lazily.\n    \"\"\"\n    try:\n        from peft import LoraConfig, get_peft_model\n    except Exception as exc:\n        raise ImportError(\"PEFT is required for EXP-4 LoRA. Install with `pip install peft`.\") from exc\n\n    base = CodeSequenceClassifier(\n        model_name=model_name,\n        freeze_backbone=False,\n        pooling=\"mean\",\n        dtype_policy=dtype_policy,\n        hf_cache_dir=hf_cache_dir,\n    )\n\n    target_modules = infer_lora_target_modules(base)\n\n    config = LoraConfig(\n        r=rank,\n        lora_alpha=lora_alpha,\n        target_modules=target_modules,\n        lora_dropout=lora_dropout,\n        bias=\"none\",\n        task_type=\"FEATURE_EXTRACTION\",\n    )\n    return get_peft_model(base, config)\n\n\ndef get_exp4_lora_model(\n    model_name: str = DEFAULT_CODE_MODEL,\n    rank: int = 8,\n    lora_alpha: int = 16,\n    dtype_policy: str = \"auto\",\n    hf_cache_dir: Optional[str] = None,\n):\n    return create_lora_sequence_classifier(\n        model_name=model_name,\n        rank=rank,\n        lora_alpha=lora_alpha,\n        dtype_policy=dtype_policy,\n        hf_cache_dir=hf_cache_dir,\n    )\n")
exp3_path.write_text("\"\"\"\nEXP-3: CodeBERTa-small-v1 frozen-encoder linear probe (Case Study 2).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import dataclass, asdict, field\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom tqdm.auto import tqdm\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.metrics import average_precision_score, precision_recall_curve, confusion_matrix\nfrom sklearn.preprocessing import StandardScaler\nimport matplotlib.pyplot as plt\nimport joblib\n\nfrom case_study_2.data_loader import create_dataloader\nfrom case_study_2.models import (\n    DEFAULT_CODE_MODEL,\n    DEFAULT_CODE_TOKENIZER,\n    configure_huggingface_cache,\n    load_code_tokenizer,\n    load_code_encoder,\n    mean_pool_last_hidden,\n    cls_pool_last_hidden,\n)\n\n# Shared, experiment-agnostic modules -- same ones EXP-0/1/2 use.\nfrom case_study_1 import split_manifest\nfrom case_study_1 import evaluation\nfrom case_study_1 import confidence_intervals\n\n\n# --------------------------------------------------------------------------- #\n# Configs\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass Exp3Config:\n    \"\"\"Static, non-tuned configuration for the EXP-3 linear probe.\"\"\"\n\n    experiment_name: str = \"cs2_exp3_codeberta_linear_probe\"\n\n    code_column: str = \"normalized_code\"\n    source_id_column: str = \"source_row_id\"\n    label_column: str = \"label\"\n    project_column: str = \"project\"\n    fold_column: str = \"fold\"\n\n    model_name: str = DEFAULT_CODE_MODEL\n    tokenizer_name: str = DEFAULT_CODE_TOKENIZER\n    hf_cache_dir: Optional[str] = None\n    max_length: int = 512\n    dtype_policy: str = \"bfloat16\"\n    embedding_batch_size: int = 64\n\n    # \"mean\" is the recommended default for a frozen probe (see models.py\n    # docstring); \"cls\" is kept available for comparison experiments.\n    pooling: str = \"mean\"\n\n    logistic_max_iter: int = 2000\n    logistic_solver: str = \"lbfgs\"\n    class_weight: str = \"balanced\"\n    decision_threshold: float = 0.50\n\n    random_state: int = 42\n    verbose: bool = True\n\n\n@dataclass\nclass NestedProbeConfig:\n    \"\"\"Nested-CV tuning configuration, mirrors NestedAlphaConfig in EXP-0.\"\"\"\n\n    experiment_name: str = \"cs2_exp3_nested_probe_dev_grouped\"\n    C_grid: Tuple[float, ...] = (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0)\n    inner_n_splits: int = 3\n    inner_random_state: int = 20260707\n    selection_metric: str = \"average_precision_pr_auc\"\n    decision_threshold: float = 0.50\n    tie_break_rule: str = \"higher_C_then_grid_order\"\n    verbose: bool = True\n\n\ndef resolve_device(require_cuda: bool = True) -> str:\n    if torch.cuda.is_available():\n        return \"cuda\"\n    if require_cuda:\n        raise RuntimeError(\n            \"EXP-3 linear probe requires a CUDA GPU for CodeBERTa embedding \"\n            \"extraction. No CUDA device was found.\"\n        )\n    return \"cpu\"\n\n\n# --------------------------------------------------------------------------- #\n# Embedding extraction (the only step that touches the GPU/encoder)\n# --------------------------------------------------------------------------- #\n\n@torch.no_grad()\ndef extract_embeddings(\n    encoder,\n    tokenizer,\n    frame: pd.DataFrame,\n    config: Exp3Config,\n    device: str,\n    cache_path: Optional[Path] = None,\n    checkpoint_every: int = 100,\n) -> np.ndarray:\n    \"\"\"\n    Run the frozen CodeBERTa encoder once over `frame` and return pooled\n    embeddings as a (n_rows, hidden_size) float32 numpy array, in the same\n    row order as `frame`. Cached to `cache_path` (.npy) if provided.\n\n    Uses length bucketing (sorts by code length before batching, then\n    un-sorts the result) to reduce wasted padding, and checkpoints\n    progress every `checkpoint_every` batches so a crashed/expired Colab\n    session can resume without redoing work already done.\n    \"\"\"\n    if cache_path is not None and cache_path.exists():\n        if config.verbose:\n            print(f\"  [embed] Loading cached embeddings: {cache_path}\")\n        return np.load(cache_path)\n\n    if device != \"cuda\":\n        raise RuntimeError(\"extract_embeddings must run on a CUDA device.\")\n\n    frame = frame.reset_index(drop=True)\n    code_lengths = frame[config.code_column].fillna(\"\").astype(str).str.len()\n    sort_order = code_lengths.sort_values(kind=\"mergesort\").index.to_numpy()\n    sorted_frame = frame.iloc[sort_order].reset_index(drop=True)\n    inverse_order = np.argsort(sort_order)\n\n    n_rows = len(sorted_frame)\n    n_batches_total = -(-n_rows // config.embedding_batch_size)\n\n    checkpoint_path = cache_path.with_suffix(\".checkpoint.npz\") if cache_path is not None else None\n    all_embeddings: List[np.ndarray] = []\n    start_batch = 0\n\n    if checkpoint_path is not None and checkpoint_path.exists():\n        ckpt = np.load(checkpoint_path)\n        all_embeddings = [ckpt[\"embeddings\"]]\n        start_batch = int(ckpt[\"n_batches\"])\n        if config.verbose:\n            print(f\"  [embed] Resuming from checkpoint: {start_batch}/{n_batches_total} batches already done\")\n\n    loader = create_dataloader(\n        sorted_frame,\n        tokenizer,\n        batch_size=config.embedding_batch_size,\n        max_length=config.max_length,\n        shuffle=False,\n        code_column=config.code_column,\n        label_column=config.label_column,\n        source_id_column=config.source_id_column,\n        project_column=config.project_column,\n        num_workers=2,\n    )\n\n    encoder.eval()\n    t0 = time.time()\n    pbar = tqdm(total=n_batches_total, initial=start_batch, desc=\"  [embed] batches\")\n\n    for i, batch in enumerate(loader):\n        if i < start_batch:\n            continue\n\n        input_ids = batch[\"input_ids\"].to(device, non_blocking=True)\n        attention_mask = batch[\"attention_mask\"].to(device, non_blocking=True)\n\n        with torch.amp.autocast(device_type=\"cuda\", dtype=torch.bfloat16):\n            outputs = encoder(input_ids=input_ids, attention_mask=attention_mask)\n            if config.pooling == \"cls\":\n                pooled = cls_pool_last_hidden(outputs.last_hidden_state)\n            else:\n                pooled = mean_pool_last_hidden(outputs.last_hidden_state, attention_mask)\n\n        all_embeddings.append(pooled.float().detach().cpu().numpy())\n        pbar.update(1)\n\n        if checkpoint_path is not None and (i + 1) % checkpoint_every == 0:\n            partial = np.concatenate(all_embeddings, axis=0)\n            np.savez(checkpoint_path, embeddings=partial, n_batches=i + 1)\n            if config.verbose:\n                elapsed_min = (time.time() - t0) / 60\n                print(f\"  [embed] checkpoint @ batch {i+1}/{n_batches_total} | elapsed {elapsed_min:.1f} min\")\n\n    pbar.close()\n\n    embeddings_sorted = np.concatenate(all_embeddings, axis=0)\n    embeddings = embeddings_sorted[inverse_order]\n\n    if config.verbose:\n        print(f\"  [embed] Extracted {embeddings.shape} in {(time.time() - t0) / 60:.2f} min (pooling={config.pooling})\")\n\n    if cache_path is not None:\n        cache_path.parent.mkdir(parents=True, exist_ok=True)\n        np.save(cache_path, embeddings)\n        if checkpoint_path is not None and checkpoint_path.exists():\n            checkpoint_path.unlink()\n\n    return embeddings\n\n\n# --------------------------------------------------------------------------- #\n# Probe fitting helpers\n# --------------------------------------------------------------------------- #\n\ndef _fit_probe(X: np.ndarray, y: np.ndarray, C: float, config: Exp3Config) -> Tuple[StandardScaler, LogisticRegression]:\n    scaler = StandardScaler(copy=False)\n    X_s = scaler.fit_transform(X.astype(np.float32, copy=False))\n    clf = LogisticRegression(\n        C=C,\n        max_iter=config.logistic_max_iter,\n        solver=config.logistic_solver,\n        class_weight=config.class_weight,\n        random_state=config.random_state,\n    )\n    clf.fit(X_s, y)\n    return scaler, clf\n\n\ndef _predict_probe(scaler: StandardScaler, clf: LogisticRegression, X: np.ndarray) -> np.ndarray:\n    return clf.predict_proba(scaler.transform(X))[:, 1]\n\n\ndef _grouped_inner_folds(\n    frame: pd.DataFrame,\n    project_column: str,\n    n_splits: int,\n    random_state: int,\n) -> List[Tuple[np.ndarray, np.ndarray]]:\n    \"\"\"Project-grouped inner K-fold split (shuffled, deterministic).\"\"\"\n    shuffled = frame.sample(frac=1.0, random_state=random_state)\n    gkf = GroupKFold(n_splits=n_splits)\n    splits = []\n    for train_pos, val_pos in gkf.split(shuffled, groups=shuffled[project_column]):\n        train_idx = shuffled.index.to_numpy()[train_pos]\n        val_idx = shuffled.index.to_numpy()[val_pos]\n        splits.append((train_idx, val_idx))\n    return splits\n\n\n# --------------------------------------------------------------------------- #\n# Nested CV (development-only, no outer holdout touched)\n# --------------------------------------------------------------------------- #\n\ndef run_exp3_nested_inner_profile(\n    development_frame: pd.DataFrame,\n    development_embeddings: np.ndarray,\n    development_manifest: pd.DataFrame,\n    outer_fold_id: int,\n    base_config: Exp3Config,\n    nested_config: NestedProbeConfig,\n) -> Dict[str, Any]:\n    \"\"\"\n    Profile a single outer-development fold: run the inner project-grouped C\n    grid search only, without producing an OOF prediction.\n    \"\"\"\n    import gc\n\n    t0 = time.time()\n\n    id_to_pos = {rid: pos for pos, rid in enumerate(development_frame[base_config.source_id_column].values)}\n\n    outer_train_ids = set(\n        development_manifest.loc[\n            development_manifest[base_config.fold_column] != outer_fold_id, base_config.source_id_column\n        ]\n    )\n    train_frame = development_frame[\n        development_frame[base_config.source_id_column].isin(outer_train_ids)\n    ].reset_index(drop=True)\n\n    inner_splits = _grouped_inner_folds(\n        train_frame, base_config.project_column, nested_config.inner_n_splits, nested_config.inner_random_state\n    )\n\n    rows = []\n    for inner_id, (tr_idx, va_idx) in enumerate(inner_splits):\n        tr_frame = train_frame.loc[tr_idx]\n        va_frame = train_frame.loc[va_idx]\n        tr_pos = [id_to_pos[rid] for rid in tr_frame[base_config.source_id_column].values]\n        va_pos = [id_to_pos[rid] for rid in va_frame[base_config.source_id_column].values]\n\n        X_tr, y_tr = development_embeddings[tr_pos], tr_frame[base_config.label_column].astype(int).values\n        X_va, y_va = development_embeddings[va_pos], va_frame[base_config.label_column].astype(int).values\n\n        for C in nested_config.C_grid:\n            scaler, clf = _fit_probe(X_tr, y_tr, C, base_config)\n            scores = _predict_probe(scaler, clf, X_va)\n            ap = average_precision_score(y_va, scores) if len(np.unique(y_va)) > 1 else float(\"nan\")\n            rows.append({\"inner_fold\": inner_id, \"C\": C, \"average_precision_pr_auc\": ap, \"n_val\": len(y_va)})\n            del scaler, clf, scores\n            gc.collect()\n\n        del tr_frame, va_frame, X_tr, X_va, y_tr, y_va\n        gc.collect()\n\n    alpha_summary = (\n        pd.DataFrame(rows)\n        .groupby(\"C\", as_index=False)[\"average_precision_pr_auc\"]\n        .mean()\n        .sort_values(\"average_precision_pr_auc\", ascending=False)\n        .reset_index(drop=True)\n    )\n    selected_C = float(alpha_summary.iloc[0][\"C\"])\n\n    del train_frame\n    gc.collect()\n\n    return {\n        \"outer_fold_id\": outer_fold_id,\n        \"selected_C\": {\"outer_fold_id\": outer_fold_id, \"selected_C\": selected_C},\n        \"C_summary\": alpha_summary,\n        \"inner_split_audit\": pd.DataFrame(rows),\n        \"total_profile_seconds\": time.time() - t0,\n    }\n\n\ndef run_exp3_nested_probe(\n    development_frame: pd.DataFrame,\n    development_embeddings: np.ndarray,\n    development_manifest: pd.DataFrame,\n    base_config: Exp3Config,\n    nested_config: NestedProbeConfig,\n    output_dir: Path,\n    additional_metadata: Optional[Dict[str, Any]] = None,\n) -> Dict[str, Any]:\n    \"\"\"\n    Official nested-CV run over all 5 frozen outer-development folds.\n    Checkpoints after every completed outer fold so a crashed/expired\n    Colab session can resume without redoing finished folds.\n    \"\"\"\n    import gc\n\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    t0 = time.time()\n\n    checkpoint_path = output_dir / \"exp3_nested_checkpoint.joblib\"\n\n    id_to_pos = {rid: pos for pos, rid in enumerate(development_frame[base_config.source_id_column].values)}\n    fold_ids = sorted(development_manifest[base_config.fold_column].unique().tolist())\n\n    oof_parts: List[pd.DataFrame] = []\n    selected_alpha_rows: List[Dict[str, Any]] = []\n    outer_training_audit: List[Dict[str, Any]] = []\n    completed_folds: set = set()\n\n    if checkpoint_path.exists():\n        ckpt = joblib.load(checkpoint_path)\n        oof_parts = ckpt[\"oof_parts\"]\n        selected_alpha_rows = ckpt[\"selected_alpha_rows\"]\n        outer_training_audit = ckpt[\"outer_training_audit\"]\n        completed_folds = ckpt[\"completed_folds\"]\n        if nested_config.verbose:\n            print(f\"  [nested] Resuming: {len(completed_folds)}/{len(fold_ids)} outer folds already completed\")\n\n    for outer_fold_id in fold_ids:\n        if outer_fold_id in completed_folds:\n            if nested_config.verbose:\n                print(f\"  [nested] Outer fold {outer_fold_id}: already completed, skipping.\")\n            continue\n\n        fold_t0 = time.time()\n        if nested_config.verbose:\n            print(f\"  [nested] Outer fold {outer_fold_id}: inner C grid search...\")\n\n        profile = run_exp3_nested_inner_profile(\n            development_frame, development_embeddings, development_manifest,\n            outer_fold_id, base_config, nested_config,\n        )\n        selected_C = profile[\"selected_C\"][\"selected_C\"]\n        selected_alpha_rows.append(profile[\"selected_C\"])\n\n        train_ids = set(\n            development_manifest.loc[\n                development_manifest[base_config.fold_column] != outer_fold_id, base_config.source_id_column\n            ]\n        )\n        val_ids = set(\n            development_manifest.loc[\n                development_manifest[base_config.fold_column] == outer_fold_id, base_config.source_id_column\n            ]\n        )\n        if train_ids.intersection(val_ids):\n            raise RuntimeError(f\"Outer fold {outer_fold_id}: train/val ID leakage detected.\")\n\n        train_frame = development_frame[development_frame[base_config.source_id_column].isin(train_ids)]\n        val_frame = development_frame[development_frame[base_config.source_id_column].isin(val_ids)]\n\n        tr_pos = [id_to_pos[rid] for rid in train_frame[base_config.source_id_column].values]\n        va_pos = [id_to_pos[rid] for rid in val_frame[base_config.source_id_column].values]\n\n        X_tr = development_embeddings[tr_pos]\n        y_tr = train_frame[base_config.label_column].astype(int).values\n        X_va = development_embeddings[va_pos]\n\n        scaler, clf = _fit_probe(X_tr, y_tr, selected_C, base_config)\n        val_scores = _predict_probe(scaler, clf, X_va)\n\n        fold_oof = pd.DataFrame({\n            base_config.source_id_column: val_frame[base_config.source_id_column].values,\n            base_config.project_column: val_frame[base_config.project_column].values,\n            \"label\": val_frame[base_config.label_column].astype(int).values,\n            \"y_score\": val_scores,\n            \"fold\": outer_fold_id,\n        })\n        oof_parts.append(fold_oof)\n\n        outer_training_audit.append({\n            \"outer_fold_id\": outer_fold_id,\n            \"selected_C\": selected_C,\n            \"n_train\": int(len(train_frame)),\n            \"n_val\": int(len(val_frame)),\n            \"train_projects\": int(train_frame[base_config.project_column].nunique()),\n            \"val_projects\": int(val_frame[base_config.project_column].nunique()),\n        })\n\n        completed_folds.add(outer_fold_id)\n\n        del train_frame, val_frame, X_tr, X_va, y_tr, scaler, clf, val_scores, profile\n        gc.collect()\n\n        joblib.dump({\n            \"oof_parts\": oof_parts,\n            \"selected_alpha_rows\": selected_alpha_rows,\n            \"outer_training_audit\": outer_training_audit,\n            \"completed_folds\": completed_folds,\n        }, checkpoint_path)\n\n        if nested_config.verbose:\n            fold_min = (time.time() - fold_t0) / 60\n            total_min = (time.time() - t0) / 60\n            print(\n                f\"  [nested] Outer fold {outer_fold_id} done in {fold_min:.1f} min \"\n                f\"| total {total_min:.1f} min | checkpoint saved\"\n            )\n\n    oof_predictions = pd.concat(oof_parts, axis=0).reset_index(drop=True)\n\n    eval_config = evaluation.EvaluationConfig(threshold=nested_config.decision_threshold, expected_n_folds=len(fold_ids))\n    eval_results = evaluation.evaluate_oof_predictions(oof_predictions, config=eval_config)\n\n    selected_alpha_df = pd.DataFrame(selected_alpha_rows)\n    outer_training_df = pd.DataFrame(outer_training_audit)\n\n    artifacts = {\n        \"oof_predictions\": output_dir / \"exp3_nested_oof_predictions.parquet\",\n        \"selected_C_per_fold\": output_dir / \"exp3_selected_C_per_fold.csv\",\n        \"outer_training_audit\": output_dir / \"exp3_outer_training_audit.csv\",\n        \"run_metadata\": output_dir / \"exp3_nested_run_metadata.json\",\n    }\n    oof_predictions.to_parquet(artifacts[\"oof_predictions\"], index=False)\n    selected_alpha_df.to_csv(artifacts[\"selected_C_per_fold\"], index=False)\n    outer_training_df.to_csv(artifacts[\"outer_training_audit\"], index=False)\n\n    metadata = {\n        \"base_config\": asdict(base_config),\n        \"nested_config\": {**asdict(nested_config), \"C_grid\": list(nested_config.C_grid)},\n        \"runtime_seconds\": time.time() - t0,\n        **(additional_metadata or {}),\n    }\n    with open(artifacts[\"run_metadata\"], \"w\", encoding=\"utf-8\") as f:\n        json.dump(metadata, f, indent=2, default=str)\n\n    if checkpoint_path.exists():\n        checkpoint_path.unlink()\n\n    return {\n        \"oof_predictions\": oof_predictions,\n        \"evaluation\": eval_results,\n        \"selected_alpha\": selected_alpha_df,\n        \"outer_fold_training\": outer_training_df,\n        \"artifacts\": artifacts,\n    }\n\n\n# --------------------------------------------------------------------------- #\n# Canonical retraining + frozen outer-holdout scoring\n# --------------------------------------------------------------------------- #\n\ndef run_exp3_canonical_retrain(\n    development_frame: pd.DataFrame,\n    development_embeddings: np.ndarray,\n    selected_C: float,\n    base_config: Exp3Config,\n    output_dir: Path,\n) -> Tuple[StandardScaler, LogisticRegression]:\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    y_dev = development_frame[base_config.label_column].astype(int).values\n    scaler, clf = _fit_probe(development_embeddings, y_dev, selected_C, base_config)\n\n    joblib.dump(clf, output_dir / \"final_exp3_linear_probe_model.joblib\")\n    joblib.dump(scaler, output_dir / \"final_exp3_scaler.joblib\")\n\n    return scaler, clf\n\n\ndef run_exp3_holdout_evaluation(\n    holdout_frame: pd.DataFrame,\n    holdout_embeddings: np.ndarray,\n    scaler: StandardScaler,\n    clf: LogisticRegression,\n    base_config: Exp3Config,\n    output_dir: Path,\n) -> Dict[str, Any]:\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    y_holdout = holdout_frame[base_config.label_column].astype(int).values\n    y_scores = _predict_probe(scaler, clf, holdout_embeddings)\n\n    holdout_predictions = pd.DataFrame({\n        base_config.source_id_column: holdout_frame[base_config.source_id_column].values,\n        base_config.project_column: holdout_frame[base_config.project_column].values,\n        \"label\": y_holdout,\n        \"y_score\": y_scores,\n        \"fold\": 0,\n    })\n    holdout_predictions.to_csv(output_dir / \"exp3_holdout_predictions.csv\", index=False)\n\n    eval_config = evaluation.EvaluationConfig(threshold=base_config.decision_threshold, expected_n_folds=1)\n    holdout_metrics = evaluation.evaluate_oof_predictions(holdout_predictions, config=eval_config)\n    print(evaluation.format_metric_report(holdout_metrics[\"pooled_metrics\"]))\n\n    precision, recall, _ = precision_recall_curve(y_holdout, y_scores)\n    ap = holdout_metrics[\"pooled_metrics\"][\"average_precision_pr_auc\"]\n    plt.figure(figsize=(6, 5))\n    plt.plot(recall, precision, color=\"b\", label=f\"EXP-3 CodeBERTa Linear Probe (PR-AUC = {ap:.4f})\")\n    plt.xlabel(\"Recall\")\n    plt.ylabel(\"Precision\")\n    plt.title(\"Precision-Recall Curve - Frozen Outer Holdout\")\n    plt.legend(loc=\"lower left\")\n    plt.grid(True)\n    plt.savefig(output_dir / \"exp3_outer_holdout_pr_curve.png\")\n    plt.close()\n\n    y_pred = (y_scores >= base_config.decision_threshold).astype(int)\n    cm = confusion_matrix(y_holdout, y_pred)\n    plt.figure(figsize=(4, 4))\n    plt.imshow(cm, cmap=plt.cm.Blues)\n    plt.title(\"Confusion Matrix - Frozen Outer Holdout\")\n    plt.xlabel(\"Predicted\")\n    plt.ylabel(\"Actual\")\n    plt.xticks([0, 1], [\"Non-Vuln (0)\", \"Vuln (1)\"])\n    plt.yticks([0, 1], [\"Non-Vuln (0)\", \"Vuln (1)\"])\n    for i in range(2):\n        for j in range(2):\n            plt.text(j, i, str(cm[i, j]), ha=\"center\", va=\"center\")\n    plt.tight_layout()\n    plt.savefig(output_dir / \"exp3_outer_holdout_confusion_matrix.png\")\n    plt.close()\n\n    return {\n        \"holdout_predictions\": holdout_predictions,\n        \"holdout_metrics\": holdout_metrics,\n        \"y_pred\": y_pred,\n    }\n")

print("Patched:", models_path)
print("Patched:", exp3_path)


## 6. Verify files and import project modules

In [ ]:
required_repo_files = [
    SRC_DIR / "case_study_1" / "split_manifest.py",
    SRC_DIR / "case_study_1" / "evaluation.py",
    SRC_DIR / "case_study_1" / "confidence_intervals.py",
    SRC_DIR / "case_study_2" / "data_loader.py",
    SRC_DIR / "case_study_2" / "models.py",
    SRC_DIR / "case_study_2" / "exp3" / "exp3_linear_probe.py",
]
missing = [str(p) for p in required_repo_files if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

import sys
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2"):
        del sys.modules[mod_name]

from case_study_2.data_loader import create_dataloader
from case_study_2.models import (
    DEFAULT_CODE_MODEL, DEFAULT_CODE_TOKENIZER, configure_huggingface_cache,
    load_code_tokenizer, load_code_encoder,
)
from case_study_2.exp3.exp3_linear_probe import (
    Exp3Config, NestedProbeConfig, resolve_device, extract_embeddings,
    run_exp3_nested_inner_profile, run_exp3_nested_probe,
    run_exp3_canonical_retrain, run_exp3_holdout_evaluation,
)
from case_study_1 import split_manifest, evaluation, confidence_intervals

print("Imported successfully.")
print("Model:", DEFAULT_CODE_MODEL, "| Tokenizer:", DEFAULT_CODE_TOKENIZER)

## 7. Config, dataset & manifest loading

In [ ]:
import pandas as pd
import subprocess

base_config = Exp3Config(hf_cache_dir=HF_CACHE_DIR, C_grid=None) if False else Exp3Config(hf_cache_dir=HF_CACHE_DIR)
nested_config = NestedProbeConfig(C_grid=C_GRID)

DEVICE = resolve_device(require_cuda=True)

if not NORMALIZED_PARQUET.is_file():
    raise FileNotFoundError(f"Missing dataset: {NORMALIZED_PARQUET}")
if not OUTER_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing outer manifest: {OUTER_MANIFEST_PATH}")
if not INNER_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing inner manifest: {INNER_MANIFEST_PATH}")

full_df = pd.read_parquet(NORMALIZED_PARQUET)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
inner_manifest_df = pd.read_parquet(INNER_MANIFEST_PATH)

required_columns = [base_config.source_id_column, base_config.code_column, base_config.label_column, base_config.project_column]
missing_columns = [c for c in required_columns if c not in full_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in dataset: {missing_columns}")

for name, frame in [("full_df", full_df), ("outer_manifest_df", outer_manifest_df), ("inner_manifest_df", inner_manifest_df)]:
    if frame["source_row_id"].duplicated().any():
        raise RuntimeError(f"{name} contains duplicate source_row_id values.")

full_indexed = full_df.set_index("source_row_id", drop=False)
dev_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "development", "source_row_id"].tolist())
holdout_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "outer_holdout", "source_row_id"].tolist())
inner_ids = set(inner_manifest_df["source_row_id"].tolist())

dev_projects = set(full_indexed.loc[list(dev_ids), base_config.project_column])
holdout_projects = set(full_indexed.loc[list(holdout_ids), base_config.project_column])
if dev_projects.intersection(holdout_projects):
    raise RuntimeError("Project leakage detected between development and outer holdout partitions.")

development_frame = full_indexed.loc[list(dev_ids)].copy().reset_index(drop=True)
holdout_frame = full_indexed.loc[list(holdout_ids)].copy().reset_index(drop=True)

print("development_frame:", len(development_frame), "rows")
print("holdout_frame:", len(holdout_frame), "rows")

## 8. Load frozen CodeBERTa encoder and extract embeddings

Length-bucketed + checkpointed extraction (resumable if the session crashes/expires). Cached to Drive, so a second run of this cell is near-instant.

In [ ]:
configure_huggingface_cache(HF_CACHE_DIR)
tokenizer = load_code_tokenizer(base_config.tokenizer_name, hf_cache_dir=HF_CACHE_DIR)

print(f"Loading frozen CodeBERTa encoder on {DEVICE}; dtype_policy={base_config.dtype_policy}; pooling={base_config.pooling}")
encoder = load_code_encoder(
    base_config.model_name, dtype_policy=base_config.dtype_policy,
    device=DEVICE, freeze=True, hf_cache_dir=HF_CACHE_DIR,
)

development_embeddings = extract_embeddings(
    encoder, tokenizer, development_frame, base_config, DEVICE,
    cache_path=EMBEDDING_CACHE_DIR / "development_embeddings.npy",
)
holdout_embeddings = extract_embeddings(
    encoder, tokenizer, holdout_frame, base_config, DEVICE,
    cache_path=EMBEDDING_CACHE_DIR / "holdout_embeddings.npy",
)

print("Development embeddings:", development_embeddings.shape)
print("Holdout embeddings:", holdout_embeddings.shape)

del encoder
torch.cuda.empty_cache()

## 9. Inner-profile smoke test (single outer fold)

In [ ]:
if RUN_PROFILE:
    profile = run_exp3_nested_inner_profile(
        development_frame=development_frame,
        development_embeddings=development_embeddings,
        development_manifest=inner_manifest_df,
        outer_fold_id=4,
        base_config=base_config,
        nested_config=nested_config,
    )
    print("Profile duration minutes:", profile["total_profile_seconds"] / 60)
    print("Selected C:")
    display(pd.DataFrame([profile["selected_C"]]))
    print("Inner C summary:")
    display(profile["C_summary"])
else:
    print("RUN_PROFILE=False; skipping profile.")

## 10. Official nested CV (checkpointed, resumable)

In [ ]:
if RUN_NESTED_OFFICIAL:
    results = run_exp3_nested_probe(
        development_frame=development_frame,
        development_embeddings=development_embeddings,
        development_manifest=inner_manifest_df,
        base_config=base_config,
        nested_config=nested_config,
        output_dir=EXP3_OUTPUT_DIR,
        additional_metadata={
            "run_kind": "exp3_codeberta_linear_probe_nested_cv_development_only",
            "global_outer_holdout_used": False,
            "input_parquet": str(NORMALIZED_PARQUET),
            "input_column": base_config.code_column,
            "model_name": base_config.model_name,
            "pooling": base_config.pooling,
            "repo_commit": subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"]).decode().strip(),
        },
    )
    print("\nOfficial EXP-3 (CodeBERTa) nested run complete.")
    print("Pooled PR-AUC:", results["evaluation"]["pooled_metrics"]["average_precision_pr_auc"])
else:
    results = None
    print("RUN_NESTED_OFFICIAL=False; official nested run skipped.")

## 11. Canonical retrain + outer holdout evaluation

In [ ]:
if RUN_CANONICAL_RETRAIN and results is not None:
    best_C = float(results["selected_alpha"]["selected_C"].mode()[0])
    print("Refitting canonical probe with C =", best_C)
    final_scaler, final_clf = run_exp3_canonical_retrain(
        development_frame=development_frame,
        development_embeddings=development_embeddings,
        selected_C=best_C,
        base_config=base_config,
        output_dir=EXP3_OUTPUT_DIR,
    )
else:
    final_scaler, final_clf = None, None
    print("RUN_CANONICAL_RETRAIN=False or no nested results available; skipping.")

if RUN_HOLDOUT_EVAL and final_clf is not None:
    holdout_results = run_exp3_holdout_evaluation(
        holdout_frame=holdout_frame,
        holdout_embeddings=holdout_embeddings,
        scaler=final_scaler,
        clf=final_clf,
        base_config=base_config,
        output_dir=EXP3_OUTPUT_DIR,
    )
    holdout_predictions_df = holdout_results["holdout_predictions"]
    holdout_metrics = holdout_results["holdout_metrics"]
else:
    holdout_results = None
    print("RUN_HOLDOUT_EVAL=False or no canonical model available; skipping.")

## 12. Cleanup

In [ ]:
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")